# Unit 10 - Analysis (Exercise)

**Atoms:** `U10-A1`, `U10-A2`, `U10-A4` · **Runtime:** ~25 seconds

## Without code
Expect p < 0.05 for the revenue `ATE`; a bootstrap CI of similar width to the t-based one; a correct ratio-of-sums `CTR` lift several times larger than the mean-of-ratios lift; and an `ATE` that still sits **below** the $5 ship threshold - statistically real, commercially not worth shipping.

## 1. The question
Complete inference, bootstrap, ratio fix, and ship decision.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

In [ ]:
n = 2000
D = np.repeat([0,1], n//2)
rev = np.concatenate([np.random.lognormal(3.4,0.5,n//2), np.random.lognormal(3.45,0.5,n//2)])
df = pd.DataFrame({'D_i': D, 'revenue': rev})
# One user in ten is a heavy user: ~2000 impressions, low CTR. Treatment helps only them.
heavy = np.random.binomial(1, 0.10, n)
imp = np.maximum(np.where(heavy==1, np.random.poisson(2000, n), np.random.poisson(20, n)), 1)
base_ctr = np.where(heavy==1, 0.02, 0.15)
treat_gain = np.where((D==1) & (heavy==1), 0.01, 0.0)
clk = np.random.binomial(imp, np.clip(base_ctr + treat_gain, 0, 1))
ratio_df = pd.DataFrame({'D_i': D, 'clicks': clk, 'impressions': imp})

## 4. TODO - t-test and CI
Return `ate`, `ci_low`, `ci_high`, `pval`. Assert p < 0.05.

In [ ]:
# TODO
ate = ci_low = ci_high = pval = None
assert pval is not None and pval < 0.05
print('ATE', round(ate,2), 'CI', (round(ci_low,2), round(ci_high,2)), 'p', round(pval,4))

## 5. TODO - bootstrap CI
500 draws. Assert CI contains `ate`.

In [ ]:
def bootstrap_ate(data, treat_col, outcome, n_boot=500):
    ates = []
    for _ in range(n_boot):
        samp = data.sample(len(data), replace=True)
        m = samp.groupby(treat_col)[outcome].mean()
        ates.append(m[1] - m[0])
    return np.percentile(ates, [2.5, 97.5])

# TODO: boot_ci = bootstrap_ate(df, 'D_i', 'revenue')
boot_ci = None
assert boot_ci is not None
assert boot_ci[0] <= ate <= boot_ci[1]
print('Bootstrap CI', np.round(boot_ci,2))

## 6. TODO - ratio CTR
Compute the treatment-minus-control `CTR` gap the **correct** way - total clicks over total impressions per arm, not the average of per-user ratios. Assert > 0. For comparison, also compute the mean-of-ratios version and see how much smaller the lift looks.

In [ ]:
# TODO
ctr_gap = None
assert ctr_gap is not None and ctr_gap > 0
print('Correct CTR gap:', round(ctr_gap,4))

## 7. TODO - practical significance
Finance will only ship if mean order value rises by $5. Set `would_ship` from the `ATE` and the threshold - and notice that a result with `p < 0.05` can still be a no.

In [ ]:
ship_threshold = 5.0
# TODO
would_ship = None
assert would_ship is not None
assert would_ship == False
print('Would ship?', would_ship)

**Takeaway:** Significant is not synonymous with ship. **Unit:** [V1](../V1/units/unit-10-analysis-decision-and-ethics/README.md) · [V2](../V2/units/unit-10-analysis-decision-and-ethics/README.md)

## Hints
- `stats.ttest_ind` on revenue by arm.
- Bootstrap: resample dataframe, difference of means.
- CTR: `sum(clicks)/sum(impressions)` per arm.

## Spoiler

```python
ctrl = df[df.D_i == 0]['revenue']
trt = df[df.D_i == 1]['revenue']
tstat, pval = stats.ttest_ind(trt, ctrl, equal_var=False)
ate = trt.mean() - ctrl.mean()
se = np.sqrt(trt.var(ddof=1) / len(trt) + ctrl.var(ddof=1) / len(ctrl))
ci_low, ci_high = ate - 1.96 * se, ate + 1.96 * se

boot_ci = bootstrap_ate(df, 'D_i', 'revenue')

totals = ratio_df.groupby('D_i')[['clicks', 'impressions']].sum()
ctr_right = totals['clicks'] / totals['impressions']
ctr_gap = ctr_right[1] - ctr_right[0]

would_ship = bool(ate >= ship_threshold)
```